In [1]:
import sys
# sys.path.append("../../")

import pandas as pd
from ext_kinematic_casadi import Ekinematic
import f110
import numpy as np

# Exploration not very IMP

In [2]:
# Initialize the model with parameters
params = f110.F110()
# Initialize the kinematic model with the parameters
model = Ekinematic(params)
# Define the sampling time
Ts = 0.02
SAMPLING_TIME = Ts
# Define the map index and name
map_idx = 0
MAPS = ['YasMarina', 'Shanghai', 'Sepang']
map_name = MAPS[map_idx]
# mu = 0.5
mu = 1.0489

In [3]:
# Friction value used for this exploratory run
mu = 1.0489
print(f"mu = {mu}")

mu = 1.0489


In [4]:
# Load the canonical raw CSV (contains C_alpha_smooth column)
df = pd.read_csv(f'../../data/raw/traj_MPPI_{map_name}_mu_{mu}.csv', comment='#')

# Drop rows where C_alpha_smooth is NaN (typically the first few rows before window fills)
df = df.dropna(subset=['C_alpha_smooth']).reset_index(drop=True)

# Getting the next state values
df['S_x_next'] = df['S_x'].shift(-1)
df['S_y_next'] = df['S_y'].shift(-1)
df['S_delta_next'] = df['S_delta'].shift(-1)
df['S_vel_next'] = df['S_vel'].shift(-1)
df['S_psi_next'] = df['S_psi'].shift(-1)
df['S_omega_next'] = df['S_omega'].shift(-1)
df['S_beta_next'] = df['S_beta'].shift(-1)
# Removing the last row as it contains NaN values for the next state
df = df[:-1]

### COMPUTING THE PREDICTED NEXT VALUES USING EKIN MODEL

In [5]:
# We compute the next state values using ekinematic model. The difference between the next state values will be used as 'target' for GP mdoel.
pred_xs, pred_ys, pred_deltas, pred_vels, pred_psis, pred_omegas, pred_betas = [], [], [], [], [], [], []
for i in range(len(df) ):
    s = df.iloc[i][['S_x', 'S_y', 'S_delta', 'S_vel', 'S_psi', 'S_omega', 'S_beta']].values
    u = df.iloc[i][['A_a', 'A_sv']].values

    dxdt = model.casadi(s, u)

    pred_x = s[0] + (dxdt[0]*SAMPLING_TIME)
    pred_y = s[1] + (dxdt[1]*SAMPLING_TIME)
    pred_delta = s[2] + (dxdt[2]*SAMPLING_TIME)
    pred_vel = s[3] + (dxdt[3]*SAMPLING_TIME)
    pred_psi = s[4] + (dxdt[4]*SAMPLING_TIME)
    pred_omega = s[5] + (dxdt[5]*SAMPLING_TIME)
    pred_beta = s[6] + (dxdt[6]*SAMPLING_TIME)
    
    if pred_psi > 2 * np.pi:
        pred_psi -= 2 * np.pi
    elif pred_psi < 0:
        pred_psi += 2 * np.pi

    pred_xs.append(pred_x)
    pred_ys.append(pred_y)
    pred_deltas.append(pred_delta)
    pred_vels.append(pred_vel)
    pred_psis.append(pred_psi)
    pred_omegas.append(pred_omega)
    pred_betas.append(pred_beta)

df['pred_x'] = pred_xs
df['pred_y'] = pred_ys
df['pred_delta'] = pred_deltas
df['pred_vel'] = pred_vels
df['pred_psi'] = pred_psis
df['pred_omega'] = pred_omegas
df['pred_beta'] = pred_betas


# Get the MSE for each state variable - Confirming that the disconnect is present in omega and beta and not in the others.
mse_x = ((df['S_x_next'] - df['pred_x']) ** 2).mean()
mse_y = ((df['S_y_next'] - df['pred_y']) ** 2).mean()
mse_delta = ((df['S_delta_next'] - df['pred_delta']) ** 2).mean()
mse_vel = ((df['S_vel_next'] - df['pred_vel']) ** 2).mean()
mse_psi = ((df['S_psi_next'] - df['pred_psi']) ** 2).mean()
mse_omega = ((df['S_omega_next'] - df['pred_omega']) ** 2).mean()
mse_beta = ((df['S_beta_next'] - df['pred_beta']) ** 2).mean()
print(f"MSE for each state variable: {mse_x.round(15)}, {mse_y.round(15)}, {mse_delta.round(6)}, {mse_vel.round(15)}, {mse_psi.round(15)}, {mse_omega}, {mse_beta}") # This confirms that the disconnect is present in omega and beta, not in the others.


MSE for each state variable: 0.0, 0.0, 1e-06, 9.063514548e-06, 4.21012e-10, 0.49580765359625373, 0.009227505776460351


In [6]:
# Compute the difference in omega and beta values that is the y columns for the GP models
df['Y_x'] = df['S_x_next'] - df['pred_x']
df['Y_y'] = df['S_y_next'] - df['pred_y']
df['Y_delta'] = df['S_delta_next'] - df['pred_delta']
df['Y_vel'] = df['S_vel_next'] - df['pred_vel']
df['Y_psi'] = df['S_psi_next'] - df['pred_psi']
df['Y_omega'] = df['S_omega_next'] - df['pred_omega']
df['Y_beta'] = df['S_beta_next'] - df['pred_beta']
# Remove columns ['S_x_next', 'S_y_next', 'S_delta_next', 'S_vel_next', 'S_psi_next', 'pred_x', 'pred_y', 'pred_delta', 'pred_vel', 'pred_psi'] from df
df.drop(columns=['S_x_next', 'S_y_next', 'S_delta_next', 'S_vel_next', 'S_psi_next', 'pred_x', 'pred_y', 'pred_delta', 'pred_vel', 'pred_psi'], inplace=True)
# Remove the columns ['S_omega_next', 'S_beta_next', 'pred_omega', 'pred_beta'] from df
df.drop(columns=['S_omega_next', 'S_beta_next', 'pred_omega', 'pred_beta'], inplace=True)

In [7]:
varidx = 6
# 7 features: 6 dynamic + C_alpha_smooth (matches prepare_dataset.load_data)
X = df[['S_delta', 'S_vel', 'S_omega', 'S_beta', 'A_a', 'A_sv', 'C_alpha_smooth']].values
# Look up Y column by name instead of fragile iloc indexing
Y_NAMES = ['Y_x', 'Y_y', 'Y_delta', 'Y_vel', 'Y_psi', 'Y_omega', 'Y_beta']
y = df[Y_NAMES[varidx]].values

from sklearn.preprocessing import StandardScaler

xscaler = StandardScaler()
yscaler = StandardScaler()
xscaler.fit(X)
yscaler.fit(y.reshape(-1, 1))
X, y = xscaler.transform(X), yscaler.transform(y.reshape(-1, 1))

In [10]:
SAMPLING_TIME

0.02

In [11]:
from prepare_dataset import load_data

X,y, xscaler, yscaler = load_data(map_name, varidx, SAMPLING_TIME, mu=mu, xscaler=None, yscaler=None)

Loading data from: ../../data/raw/traj_MPPI_YasMarina_mu_1.0489.csv
MSE for each state variable: 0.0, 0.0, 6.72108813136627e-07, 9.0635e-06, 4e-10, 0.49580765359625373, 0.009227505776460351


# Main Splits generation 

In [17]:
from prepare_dataset import load_data
from sklearn.model_selection import train_test_split
import os
import pickle
import json
import os, json, csv, time, datetime as dt, _pickle as pickle
import numpy as np
# -----------------------------
# Config
# -----------------------------
Ts = 0.02
SAMPLING_TIME = Ts
map_idx = 0
MAPS = ['YasMarina', 'Shanghai', 'Sepang', 'Austin', 'BrandsHatch', 'IMS', 'YasMarina_10489']
map_name = MAPS[map_idx]
VARIDX = 6 # 0: x, 1: y, 2: delta, 3: vel, 4: psi, 5: omega, 6: beta
NUM_TRIALS = 170
SEED = 42

# Output root for generated splits (relative to src/data_preprocess/).
# Splits land in <repo>/data/splits/  ; reference splits live in <repo>/data/splits_verify/
SPLITS_OUTPUT_ROOT = "../../data/splits"

# mu = 0.5 # The friction value
# mu = 0.8 # The friction value
# mu = 1.0 # The friction value
# mu = 1.0489 # The friction value
# mu = 0.8642 # The friction value
mu = 0.4947 # The friction value

# -----------------------------
# Load & split 0.5 / 0.2 / 0.3
# -----------------------------
# if mu != 1.0489:
#     # Loading the scalers from the original dataset
#     xscaler = pickle.load(open(f'{SPLITS_OUTPUT_ROOT}/{map_name}_VAR{VARIDX}_Ts{SAMPLING_TIME:.3f}/xscaler.pkl', 'rb'))
#     yscaler = pickle.load(open(f'{SPLITS_OUTPUT_ROOT}/{map_name}_VAR{VARIDX}_Ts{SAMPLING_TIME:.3f}/yscaler.pkl', 'rb'))
# else:
xscaler = None
yscaler = None

X, y, xscaler, yscaler = load_data(map_name, VARIDX, SAMPLING_TIME, xscaler=xscaler, yscaler=yscaler, mu=mu)

# Did this to compare against the meta learning results
X_train, y_train = X[:1737], y[:1737]
X_val, y_val = X[1737:1737+217], y[1737:1737+217]
X_test, y_test = X[1737+217:], y[1737+217:]

# X_train, X_temp, y_train, y_temp = train_test_split(
#     X, y, test_size=0.5, random_state=SEED, shuffle=True
# )
# # 0.5 temp -> 40% val (0.2 overall), 60% test (0.3 overall)
# X_val, X_test, y_val, y_test = train_test_split(
#     X_temp, y_temp, test_size=0.6, random_state=SEED, shuffle=True
# )
print(f"Train: {X_train.shape[0]}, Val: {X_val.shape[0]}, Test: {X_test.shape[0]}")

# -----------------------------
# Save splits & scalers BEFORE Optuna
# -----------------------------
if mu == 1.0489 and map_name == 'YasMarina':
    base_dir = os.path.join(SPLITS_OUTPUT_ROOT, f"{map_name}_VAR{VARIDX}_Ts{SAMPLING_TIME:.3f}")
else:
    base_dir = os.path.join(SPLITS_OUTPUT_ROOT, f"{map_name}_VAR{VARIDX}_Ts{SAMPLING_TIME:.3f}_mu_{mu}")
os.makedirs(base_dir, exist_ok=True)

np.savez_compressed(
    os.path.join(base_dir, "splits.npz"),
    X_train=X_train, y_train=y_train,
    X_val=X_val, y_val=y_val,
    X_test=X_test, y_test=y_test,
    x_scaler_mean=xscaler.mean_,
    x_scaler_std=xscaler.scale_,
    y_scaler_mean=yscaler.mean_,
    y_scaler_std=yscaler.scale_,
)

with open(os.path.join(base_dir, "xscaler.pkl"), "wb") as f:
    pickle.dump(xscaler, f)
with open(os.path.join(base_dir, "yscaler.pkl"), "wb") as f:
    pickle.dump(yscaler, f)

with open(os.path.join(base_dir, "meta.json"), "w") as f:
    json.dump(
        {
            "map_name": map_name,
            "VARIDX": VARIDX,
            "Ts": SAMPLING_TIME,
            "split": {"train": 0.5, "val": 0.2, "test": 0.3},
            "sizes": {
                "X": int(X.shape[0]),
                "train": int(X_train.shape[0]),
                "val": int(X_val.shape[0]),
                "test": int(X_test.shape[0]),
            },
            "saved_at": dt.datetime.now().isoformat()
        },
        f, indent=2
    )

Loading data from: ../../data/raw/traj_MPPI_YasMarina_mu_0.4947.csv
MSE for each state variable: 0.0, 0.0, 4.286926874415145e-06, 5.604e-07, 3e-10, 0.21527709043493115, 0.004971813531156265
Train: 1737, Val: 217, Test: 1037


In [18]:
# Undo scaling: convert scaled data back to original space
# For X: x_original = xscaler.inverse_transform(X_scaled)
# For y: y_original = yscaler.inverse_transform(y_scaled)

X_train_original = xscaler.inverse_transform(X_train)
y_train_original = yscaler.inverse_transform(y_train)

X_val_original = xscaler.inverse_transform(X_val)
y_val_original = yscaler.inverse_transform(y_val)

X_test_original = xscaler.inverse_transform(X_test)
y_test_original = yscaler.inverse_transform(y_test)

print(f"X_train scaled range:   [{X_train.min():.3f}, {X_train.max():.3f}]")
print(f"X_train original range: [{X_train_original.min():.3f}, {X_train_original.max():.3f}]")
print(f"y_train scaled range:   [{y_train.min():.3f}, {y_train.max():.3f}]")
print(f"y_train original range: [{y_train_original.min():.3f}, {y_train_original.max():.3f}]")

X_train scaled range:   [-3.403, 13.619]
X_train original range: [-9.510, 97.382]
y_train scaled range:   [-4.583, 4.775]
y_train original range: [-0.323, 0.336]


In [19]:
X_train scaled range:   [-9.293, 4.643]
X_train original range: [-9.510, 104.890]
y_train scaled range:   [-3.585, 3.482]
y_train original range: [-0.345, 0.334]

SyntaxError: invalid syntax (3659975115.py, line 1)